In [1]:
from io import StringIO
import boto3
import pandas as pd
import numpy as np
import scipy.stats as stats
from datetime import datetime

In [2]:
nbm_s3_url = "s3://processed-data-809918852303-us-east-1-an/nbm_qpf24pmean_DEPLOYMENT.parquet"
nbm_df = pd.read_parquet(nbm_s3_url)

In [3]:
nbm_df['date'] = pd.to_datetime(nbm_df['date'])
nbm_df = nbm_df.sort_values(by=['public_zone', 'date', 'week', 'forecast_day'])

In [4]:
prism_s3_url = 's3://processed-data-809918852303-us-east-1-an/prism_qpf24pmean_DEPLOYMENT.parquet'
prism_df = pd.read_parquet(prism_s3_url)
prism_df = prism_df.sort_values(["unique_zone_str", "date"])

In [5]:
# ensure data sorted chronologically
prism_df['date'] = pd.to_datetime(prism_df['date'])
prism_df = prism_df.sort_index()

# separate wet day column
prism_df["rain_only"] = prism_df["qpf24pmean_value"].where(prism_df["qpf24pmean_value"] > 0)

In [6]:
def calc_rolling_stats(group, window_num):
    mean = group.rolling(window=window_num, min_periods=window_num).mean().shift(1)
    var = group.rolling(window=window_num, min_periods=window_num).var().shift(1)
    return pd.DataFrame({"mean": mean, "var": var}, index=group.index)

# rolling calculations, shifted by 1 day so as to not take current day into account
stats_df_60d = prism_df.groupby("global_zone_id", group_keys=False)['qpf24pmean_value'].apply(calc_rolling_stats, window_num=60)
stats_df_7d = prism_df.groupby("global_zone_id", group_keys=False)['qpf24pmean_value'].apply(calc_rolling_stats, window_num=7)

In [7]:
# Method of Moments with a protection clip against 0 variance, to prevent zero deivionn errors in dry periods
stats_df_60d["var"] = stats_df_60d["var"].replace(0, np.nan)

prism_df["gamma_shape_60d"] = (stats_df_60d["mean"] ** 2) / stats_df_60d["var"]
prism_df["gamma_scale_60d"] = stats_df_60d["var"] / stats_df_60d["mean"]

# same thing for 3 day
stats_df_7d["var"] = stats_df_7d["var"].replace(0, np.nan)

prism_df["gamma_shape_7d"] = (stats_df_7d["mean"] ** 2) / stats_df_7d["var"]
prism_df["gamma_scale_7d"] = stats_df_7d["var"] / stats_df_7d["mean"]

In [8]:
max_forecast_date = nbm_df["date"].max()
zones = prism_df["unique_zone_str"].unique()

full_idx = pd.MultiIndex.from_product(
    [zones, pd.date_range(prism_df["date"].min(), max_forecast_date)],
    names=["unique_zone_str", "date"],
)

In [9]:
prism_df_extended = (
    prism_df.set_index(["unique_zone_str", "date"])
    .reindex(full_idx)
    .groupby("unique_zone_str")
    .ffill()  # forward-fill latest data, 6 days in advance
    .reset_index()
)

In [10]:
prism_df_extended = prism_df_extended.replace([np.inf, -np.inf], np.nan).fillna(0)

In [11]:
prism_df_extended[prism_df_extended['date'] >= '2026-09-12'].isna().sum()

unique_zone_str     0
date                0
global_zone_id      0
qpf24pmean_value    0
week                0
zone_id             0
state               0
name                0
rain_only           0
gamma_shape_60d     0
gamma_scale_60d     0
gamma_shape_7d      0
gamma_scale_7d      0
dtype: int64

In [12]:
prism_df_extended

,unique_zone_str,date,global_zone_id,qpf24pmean_value,week,zone_id,state,name,rain_only,gamma_shape_60d,gamma_scale_60d,gamma_shape_7d,gamma_scale_7d
0,AL_001,2026-07-06,1.0,0.163947,28,001,AL,Lauderdale,0.163947,0.000000,0.000000,0.000000,0.000000
1,AL_001,2026-07-07,1.0,0.091287,28,001,AL,Lauderdale,0.091287,0.000000,0.000000,0.000000,0.000000
2,AL_001,2026-07-08,1.0,0.103248,28,001,AL,Lauderdale,0.103248,0.000000,0.000000,0.000000,0.000000
3,AL_001,2026-07-09,1.0,0.311628,28,001,AL,Lauderdale,0.311628,0.000000,0.000000,0.000000,0.000000
4,AL_001,2026-07-10,1.0,0.010131,28,001,AL,Lauderdale,0.010131,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
265645,WY_199,2026-09-08,3850.0,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375
265646,WY_199,2026-09-09,3850.0,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375
265647,WY_199,2026-09-10,3850.0,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375
265648,WY_199,2026-09-11,3850.0,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375


Merging

In [15]:
# merge prism_df and nbm_df on date and zone string to ensure match

merged_df = pd.merge(
    nbm_df, 
    prism_df_extended, 
    on=['date', 'unique_zone_str'], 
    suffixes=('_nbm', '_prism'),
    how='left'
)

In [16]:
merged_df

,public_zone,date,week_nbm,qpf24pmean_value_nbm,forecast_day,unique_zone_str,model_run_date,zone_id_nbm,state_nbm,name_nbm,...,qpf24pmean_value_prism,week_prism,zone_id_prism,state_prism,name_prism,rain_only,gamma_shape_60d,gamma_scale_60d,gamma_shape_7d,gamma_scale_7d
0,1,2026-07-07,28,0.159420,1,AL_001,2026-07-07,001,AL,Lauderdale,...,0.091287,28,001,AL,Lauderdale,0.091287,0.000000,0.000000,0.000000,0.000000
1,1,2026-07-08,28,0.155903,1,AL_001,2026-07-08,001,AL,Lauderdale,...,0.103248,28,001,AL,Lauderdale,0.103248,0.000000,0.000000,0.000000,0.000000
2,1,2026-07-08,28,0.174770,2,AL_001,2026-07-07,001,AL,Lauderdale,...,0.103248,28,001,AL,Lauderdale,0.103248,0.000000,0.000000,0.000000,0.000000
3,1,2026-07-09,28,0.047548,1,AL_001,2026-07-09,001,AL,Lauderdale,...,0.311628,28,001,AL,Lauderdale,0.311628,0.000000,0.000000,0.000000,0.000000
4,1,2026-07-09,28,0.103638,2,AL_001,2026-07-08,001,AL,Lauderdale,...,0.311628,28,001,AL,Lauderdale,0.311628,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1670895,3850,2026-09-10,37,0.000000,6,WY_199,2026-09-05,199,WY,Sheridan Foothills,...,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375
1670896,3850,2026-09-10,37,0.020948,7,WY_199,2026-09-04,199,WY,Sheridan Foothills,...,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375
1670897,3850,2026-09-11,37,0.064423,6,WY_199,2026-09-06,199,WY,Sheridan Foothills,...,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375
1670898,3850,2026-09-11,37,0.055256,7,WY_199,2026-09-05,199,WY,Sheridan Foothills,...,0.000018,36,199,WY,Sheridan Foothills,0.000018,0.152633,0.152023,0.175552,0.026375


In [18]:
merged_df = merged_df.drop(columns=['week_nbm', 'zone_id_nbm', 'state_nbm', 'name_nbm', 'zone_id_nbm', 'state_nbm', 'name_nbm', 
                                    'global_zone_id', 'unique_zone_str'
                                   ])

In [19]:
merged_df[merged_df['date'] >= '2026-09-12'].isna().sum()

public_zone               0
date                      0
qpf24pmean_value_nbm      0
forecast_day              0
model_run_date            0
qpf24pmean_value_prism    0
week_prism                0
zone_id_prism             0
state_prism               0
name_prism                0
rain_only                 0
gamma_shape_60d           0
gamma_scale_60d           0
gamma_shape_7d            0
gamma_scale_7d            0
dtype: int64

In [21]:
# Calculate the temp difference in F (NBM minus PRISM)
merged_df['nbm_minus_obs'] = merged_df['qpf24pmean_value_nbm'] - merged_df['qpf24pmean_value_prism']

merged_df = merged_df.rename(columns={'week_prism': 'week', 'state_prism': 'state', 'name_prism':'name', 'zone_id_prism':'zone_id'})

column_order = [
        'public_zone', 'date', 'week', 'forecast_day', 'nbm_minus_obs', 'state', 'zone_id', 'name', 
        'model_run_date', 'qpf24pmean_value_nbm', 'qpf24pmean_value_prism', 'rain_only', 'gamma_shape_60d', 'gamma_scale_60d',
        'gamma_shape_7d', 'gamma_scale_7d'
]
merged_df = merged_df[column_order]

In [22]:
merged_df

,public_zone,date,week,forecast_day,nbm_minus_obs,state,zone_id,name,model_run_date,qpf24pmean_value_nbm,qpf24pmean_value_prism,rain_only,gamma_shape_60d,gamma_scale_60d,gamma_shape_7d,gamma_scale_7d
0,1,2026-07-07,28,1,0.068133,AL,001,Lauderdale,2026-07-07,0.159420,0.091287,0.091287,0.000000,0.000000,0.000000,0.000000
1,1,2026-07-08,28,1,0.052655,AL,001,Lauderdale,2026-07-08,0.155903,0.103248,0.103248,0.000000,0.000000,0.000000,0.000000
2,1,2026-07-08,28,2,0.071522,AL,001,Lauderdale,2026-07-07,0.174770,0.103248,0.103248,0.000000,0.000000,0.000000,0.000000
3,1,2026-07-09,28,1,-0.264080,AL,001,Lauderdale,2026-07-09,0.047548,0.311628,0.311628,0.000000,0.000000,0.000000,0.000000
4,1,2026-07-09,28,2,-0.207990,AL,001,Lauderdale,2026-07-08,0.103638,0.311628,0.311628,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1670895,3850,2026-09-10,36,6,-0.000018,WY,199,Sheridan Foothills,2026-09-05,0.000000,0.000018,0.000018,0.152633,0.152023,0.175552,0.026375
1670896,3850,2026-09-10,36,7,0.020930,WY,199,Sheridan Foothills,2026-09-04,0.020948,0.000018,0.000018,0.152633,0.152023,0.175552,0.026375
1670897,3850,2026-09-11,36,6,0.064405,WY,199,Sheridan Foothills,2026-09-06,0.064423,0.000018,0.000018,0.152633,0.152023,0.175552,0.026375
1670898,3850,2026-09-11,36,7,0.055238,WY,199,Sheridan Foothills,2026-09-05,0.055256,0.000018,0.000018,0.152633,0.152023,0.175552,0.026375


In [23]:
# calculate CDF. temporarily fill NaN shapes/scales with 1 just so the math function runs,
# then overwrite the invalid days afterward.
safe_shape_60d = merged_df["gamma_shape_60d"].fillna(1)
safe_scale_60d = merged_df["gamma_scale_60d"].fillna(1)

safe_shape_7d = merged_df["gamma_shape_7d"].fillna(1)
safe_scale_7d = merged_df["gamma_scale_7d"].fillna(1)

In [24]:
merged_df["forecast_probability"] = stats.gamma.cdf(
    merged_df["qpf24pmean_value_nbm"], safe_shape_60d, scale=safe_scale_60d
)

/home/ec2-user/miniconda/lib/python3.13/site-packages/scipy/stats/_distn_infrastructure.py:2159: RuntimeWarning: divide by zero encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/home/ec2-user/miniconda/lib/python3.13/site-packages/scipy/stats/_distn_infrastructure.py:2159: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)


In [25]:
# clean up the edges and dry periods
# if there weren't enough rainy days to fit a Gamma distribution, the probability that a '0' forecast is normal is 100% (or 0 baseline)
merged_df.loc[merged_df["gamma_shape_60d"].isna(), "forecast_probability"] = 0.0

# If the forecast itself is 0, the probability is 0 (Gamma is strictly > 0)
merged_df.loc[merged_df["qpf24pmean_value_nbm"] == 0, "forecast_probability"] = 0.0

In [26]:
# expected rain amount on a wet day
merged_df["past_60d_gamma_mean"] = merged_df["gamma_shape_60d"] * merged_df["gamma_scale_60d"]

# fill NaNs with 0 for completely dry periods
merged_df["past_60d_gamma_mean"] = merged_df["past_60d_gamma_mean"].fillna(0)


# expected rain amount on a wet day
merged_df["past_7d_gamma_mean"] = merged_df["gamma_shape_7d"] * merged_df["gamma_scale_7d"]

# fill NaNs with 0 for completely dry periods
merged_df["past_7d_gamma_mean"] = merged_df["past_7d_gamma_mean"].fillna(0)

In [27]:
merged_df['nbm_minus_obs_window_60d'] = merged_df['qpf24pmean_value_nbm'] - merged_df['past_60d_gamma_mean']
merged_df['nbm_minus_obs_window_7d'] = merged_df['qpf24pmean_value_nbm'] - merged_df['past_7d_gamma_mean']

In [28]:
merged_df

,public_zone,date,week,forecast_day,nbm_minus_obs,state,zone_id,name,model_run_date,qpf24pmean_value_nbm,...,rain_only,gamma_shape_60d,gamma_scale_60d,gamma_shape_7d,gamma_scale_7d,forecast_probability,past_60d_gamma_mean,past_7d_gamma_mean,nbm_minus_obs_window_60d,nbm_minus_obs_window_7d
0,1,2026-07-07,28,1,0.068133,AL,001,Lauderdale,2026-07-07,0.159420,...,0.091287,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.00000,0.159420,0.159420
1,1,2026-07-08,28,1,0.052655,AL,001,Lauderdale,2026-07-08,0.155903,...,0.103248,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.00000,0.155903,0.155903
2,1,2026-07-08,28,2,0.071522,AL,001,Lauderdale,2026-07-07,0.174770,...,0.103248,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.00000,0.174770,0.174770
3,1,2026-07-09,28,1,-0.264080,AL,001,Lauderdale,2026-07-09,0.047548,...,0.311628,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.00000,0.047548,0.047548
4,1,2026-07-09,28,2,-0.207990,AL,001,Lauderdale,2026-07-08,0.103638,...,0.311628,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.00000,0.103638,0.103638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1670895,3850,2026-09-10,36,6,-0.000018,WY,199,Sheridan Foothills,2026-09-05,0.000000,...,0.000018,0.152633,0.152023,0.175552,0.026375,0.000000,0.023204,0.00463,-0.023204,-0.004630
1670896,3850,2026-09-10,36,7,0.020930,WY,199,Sheridan Foothills,2026-09-04,0.020948,...,0.000018,0.152633,0.152023,0.175552,0.026375,0.778775,0.023204,0.00463,-0.002256,0.016318
1670897,3850,2026-09-11,36,6,0.064405,WY,199,Sheridan Foothills,2026-09-06,0.064423,...,0.000018,0.152633,0.152023,0.175552,0.026375,0.893653,0.023204,0.00463,0.041219,0.059793
1670898,3850,2026-09-11,36,7,0.055238,WY,199,Sheridan Foothills,2026-09-05,0.055256,...,0.000018,0.152633,0.152023,0.175552,0.026375,0.878941,0.023204,0.00463,0.032052,0.050626


In [29]:
# short-term volatility, checks if error spikes right now
merged_df['nbm_minus_obs_7d_std'] = merged_df['nbm_minus_obs_window_7d'].rolling(window=7).std()

In [30]:
column_order = [
        'public_zone', 'date', 'week', 'forecast_day', 'nbm_minus_obs', 'nbm_minus_obs_window_60d', 'nbm_minus_obs_window_7d',
        'nbm_minus_obs_7d_std', 'past_60d_gamma_mean', 'past_7d_gamma_mean', 'state', 'zone_id', 'name', 'model_run_date', 'qpf24pmean_value_nbm', 
        'qpf24pmean_value_prism'
]

merged_df = merged_df[column_order]

In [31]:
merged_df = merged_df.sort_values(by=['public_zone', 'model_run_date', 'week', 'forecast_day'])

In [34]:
merged_df

,public_zone,date,week,forecast_day,nbm_minus_obs,nbm_minus_obs_window_60d,nbm_minus_obs_window_7d,nbm_minus_obs_7d_std,past_60d_gamma_mean,past_7d_gamma_mean,state,zone_id,name,model_run_date,qpf24pmean_value_nbm,qpf24pmean_value_prism
0,1,2026-07-07,28,1,0.068133,0.159420,0.159420,NaN,0.000000,0.00000,AL,001,Lauderdale,2026-07-07,0.159420,0.091287
2,1,2026-07-08,28,2,0.071522,0.174770,0.174770,NaN,0.000000,0.00000,AL,001,Lauderdale,2026-07-07,0.174770,0.103248
5,1,2026-07-09,28,3,-0.256275,0.055353,0.055353,NaN,0.000000,0.00000,AL,001,Lauderdale,2026-07-07,0.055353,0.311628
9,1,2026-07-10,28,4,0.161903,0.172035,0.172035,0.100429,0.000000,0.00000,AL,001,Lauderdale,2026-07-07,0.172035,0.010131
14,1,2026-07-11,28,5,0.081941,0.453887,0.453887,0.283193,0.000000,0.00000,AL,001,Lauderdale,2026-07-07,0.453887,0.371946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1670885,3850,2026-09-08,36,3,-0.000018,-0.023204,-0.004630,0.062827,0.023204,0.00463,WY,199,Sheridan Foothills,2026-09-06,0.000000,0.000018
1670890,3850,2026-09-09,36,4,-0.000018,-0.023204,-0.004630,0.034433,0.023204,0.00463,WY,199,Sheridan Foothills,2026-09-06,0.000000,0.000018
1670894,3850,2026-09-10,36,5,-0.000018,-0.023204,-0.004630,0.028551,0.023204,0.00463,WY,199,Sheridan Foothills,2026-09-06,0.000000,0.000018
1670897,3850,2026-09-11,36,6,0.064405,0.041219,0.059793,0.023955,0.023204,0.00463,WY,199,Sheridan Foothills,2026-09-06,0.064423,0.000018


In [36]:
BUCKET_NAME = 'processed-data-809918852303-us-east-1-an'

merged_df.to_parquet('merged_qpf24pmean_DEPLOYMENT.parquet', index=False)
s3 = boto3.client('s3')
s3.upload_file('merged_qpf24pmean_DEPLOYMENT.parquet', BUCKET_NAME, 'merged_qpf24pmean_DEPLOYMENT.parquet')
print("PARQUET complete / uploaded now")

PARQUET complete / uploaded now


done

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

plot_day = '2026-09-06'
day_diff_df = merged_df[merged_df['date'] == plot_day]

zones_gdf = gpd.read_file("https://www.weather.gov/source/gis/Shapefiles/WSOM/z_16ap26.zip")
zones_gdf['unique_zone_str'] = zones_gdf['STATE'] + '_' + zones_gdf['ZONE']
zones_gdf['STATE_ZONE'] = zones_gdf['STATE'] + '_' + zones_gdf['ZONE']

zones_display = zones_gdf.to_crs(epsg=4269)
map_gdf = zones_display.merge(day_diff_df, left_on='STATE_ZONE', right_on='unique_zone_str')

#initialize the plot figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
zones_display.plot(ax=ax, color='lightgrey', edgecolor='none')
plt.grid(linestyle='--', color='grey', alpha=0.2)

plt.xlim(-126, -65)
plt.ylim(24, 50)

max_error_bound = 10

map_gdf.plot(
    column='temp_diff_f',
    ax=ax,
    legend=True,
    legend_kwds={'label': 'temperature diff in °F', 'orientation': 'horizontal','extend': 'both', 'pad': 0.05},
    cmap='RdBu_r',
    vmin=-max_error_bound,
    vmax=max_error_bound
)

plt.title('NBM bias for ' + plot_day + ' forecast day 1', fontsize=13, fontweight='bold') # this is model - obs
plt.tight_layout()
plt.show()